[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# [模块 6](https://dataflowr.github.io/website/modules/6-convolutional-neural-network/)：通过例子学卷积

我们将从零搭建我们的第一个卷积神经网络（CNN）。


## 1. 准备工作


In [ ]:
%matplotlib inline
import math,sys,os,numpy as np
from numpy.linalg import norm
from matplotlib import pyplot as plt

In [ ]:
import torch
import torchvision
from torchvision import models,transforms,datasets

把 MNIST 数据下载到磁盘上，并转换成 pytorch 兼容的格式。

```torchvision.datasets``` 为一批常用数据集提供了（下载、格式化）支持。```torchvision``` 里可用的数据集列表可以看[这里](http://pytorch.org/docs/master/torchvision/datasets.html)。

注意下载只会执行一次。函数总是先检查数据是否已经在磁盘上。


In [ ]:
root_dir = './data/MNIST/'
torchvision.datasets.MNIST(root=root_dir,download=True)

MNIST 数据集由手写数字的小图片组成。图片是灰度图，尺寸为 28 x 28。有 60,000 张训练图片和 10,000 张测试图片。


In [ ]:
train_set = torchvision.datasets.MNIST(root=root_dir, train=True, download=True)

为已经下载到磁盘上的 MNIST 数据定义并初始化一个数据加载器。


In [ ]:
MNIST_dataset = torch.utils.data.DataLoader(train_set, batch_size=1, shuffle=True, num_workers=1)

对于当前这个 notebook，我们可以把数据格式化为 _numpy ndarrays_，这样在 matplotlib 里更容易绘图。同样的操作在 _pytorch Tensors_ 上也能轻松完成。


In [ ]:
images = train_set.data.numpy().astype(np.float32)/255
labels = train_set.targets.numpy()

In [ ]:
print(images.shape,labels.shape)

## 2. 数据可视化

为了方便，我们定义几个格式化、绘制图像数据的函数


In [ ]:
# 绘制多张图片
def plots(ims, interp=False, titles=None):
    ims=np.array(ims)
    mn,mx=ims.min(),ims.max()
    f = plt.figure(figsize=(12,24))
    for i in range(len(ims)):
        sp=f.add_subplot(1, len(ims), i+1)
        if not titles is None: sp.set_title(titles[i], fontsize=18)
        plt.imshow(ims[i], interpolation=None if interp else 'none', vmin=mn,vmax=mx)

# 绘制单张图片
def plot(im, interp=False):
    f = plt.figure(figsize=(3,6), frameon=True)
    plt.imshow(im, interpolation=None if interp else 'none')

plt.gray()
plt.close()

In [ ]:
plot(images[5000])

In [ ]:
labels[5000]

In [ ]:
plots(images[5000:5005], titles=labels[5000:5005])

## 3. 一个简单的分类器

这一节我们将构建一个基本的二分类器。
这个分类器会告诉我们一张给定的图片是 _1_ 还是 _8_。

我们取出 _8_ 类和 _1_ 类的所有图片。


In [ ]:
n=len(images)

In [ ]:
eights=[images[i] for i in range(n) if labels[i]==8]
ones=[images[i] for i in range(n) if labels[i]==1]

In [ ]:
len(eights), len(ones)

In [ ]:
plots(eights[:5])
plots(ones[:5])

测试集保留前 1000 张数字，其余所有数字取平均。


In [ ]:
raws8 =  np.mean(eights[1000:],axis=0)

plot(raws8)

我们对 1 做同样的事：


In [ ]:
raws1 =  np.mean(ones[1000:],axis=0)

plot(raws1)

我们构造了一个 8 的'典型代表'和一个 1 的'典型代表'。现在，对于测试集里的一个新样本，我们计算这个样本到两个代表之间的距离，并把样本分类到距离最近的代表的标签。

对于图像之间的距离，我们只取逐像素的平方距离。


In [ ]:
# 平方距离之和
def sse(a,b): return ((a-b)**2).sum()

# 如果最接近 8 则返回 1，否则返回 0
def is8_raw_n2(im): return 1 if sse(im,raws1) > sse(im,raws8) else 0

In [ ]:
nb_8_predicted_8, nb_1_predicted_8 = [np.array([is8_raw_n2(im) for im in ims]).sum() for ims in [eights[:1000],ones[:1000]]]

nb_8_predicted_1, nb_1_predicted_1 = [np.array([(1-is8_raw_n2(im)) for im in ims]).sum() for ims in [eights[:1000],ones[:1000]]]

# 只是验证一下
print(nb_8_predicted_1+nb_8_predicted_8, nb_1_predicted_1+nb_1_predicted_8)

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/Precisionrecall.svg/1024px-Precisionrecall.svg.png" alt="Drawing" style="width: 500px;"/>

来源 [wikipedia](https://en.wikipedia.org/wiki/Precision_and_recall)


In [ ]:
def compute_scores(nb_8_predicted_8,nb_8_predicted_1,nb_1_predicted_1,nb_1_predicted_8):
    Precision_8 = nb_8_predicted_8/(nb_8_predicted_8+nb_1_predicted_8)
    Recall_8 = nb_8_predicted_8/(nb_8_predicted_1+nb_8_predicted_8)
    Precision_1 = nb_1_predicted_1/(nb_1_predicted_1+nb_8_predicted_1)
    Recall_1 = nb_1_predicted_1/(nb_1_predicted_1+nb_1_predicted_8)
    return Precision_8, Recall_8, Precision_1, Recall_1

Precision_8, Recall_8, Precision_1, Recall_1 = compute_scores(nb_8_predicted_8,nb_8_predicted_1,nb_1_predicted_1,nb_1_predicted_8)

print('precision 8:', Precision_8, 'recall 8:', Recall_8)
print('precision 1:', Precision_1, 'recall 1:', Recall_1)
print('accuracy :', (Recall_1+Recall_8)/2)

这是我们二分类任务的基线。接下来你的任务是用卷积做得更好！


## 4. 滤波器与卷积

先看一下这个关于[交互式图像卷积核](http://setosa.io/ev/image-kernels/)的可视化解释。

在某些领域，卷积或滤波可以更好地理解为_相关_。
实践中，我们把滤波器矩阵在图像（一个更大的矩阵）上滑动，总是选取与滤波器同样大小的图像块。我们计算滤波器与图像块的点积，并把得到的标量响应保存下来，这个响应反映了滤波器与图像块之间的相似/相关程度。

下面是一个简单的 3x3 滤波器，也就是一个 3x3 矩阵（更多例子见 [Sobel 算子](https://en.wikipedia.org/wiki/Sobel_operator)）


In [ ]:
top=[[-1,-1,-1],
     [ 1, 1, 1],
     [ 0, 0, 0]]

plot(top)

现在我们创建一个玩具图像，来理解卷积是怎么运作的。


In [ ]:
cross = np.zeros((28,28))
cross += np.eye(28)
for i in range(4):
    cross[12+i,:] = np.ones(28)
    cross[:,12+i] = np.ones(28)

plot(cross)

我们的 `top` 滤波器应该突出图像中的顶部水平边缘。


In [ ]:
from scipy.ndimage.filters import convolve, correlate

corr_cross = correlate(cross,top)

plot(corr_cross)

In [ ]:
?correlate

图像边缘处是怎么处理的？

## Padding

![padding](https://dataflowr.github.io/notebooks/Module6/img/padding_conv.gif)

来源：[卷积动画](https://github.com/vdumoulin/conv_arithmetic/blob/master/README.md)


In [ ]:
# 看看 padding 的作用
corr_cross = correlate(cross,top, mode='constant')
plot(corr_cross)

In [ ]:
corrtop = correlate(images[5000], top)
plot(corrtop)

把滤波器旋转 90 度再调用 ```convolve``` 函数，得到的结果和之前调用 ```correlate``` 函数是一样的。


In [ ]:
np.rot90(top, 1)

In [ ]:
convtop = convolve(images[5000], np.rot90(top,2))
plot(convtop)
np.allclose(convtop, corrtop)

让我们为这个简单的 3x3 滤波器再生成一些变体


In [ ]:
straights=[np.rot90(top,i) for i in range(4)]
plots(straights)

我们按类似的方式生成一组行为不同的滤波器


In [ ]:
br=[[ 0, 0, 1],
    [ 0, 1,-1.5],
    [ 1,-1.5, 0]]

diags = [np.rot90(br,i) for i in range(4)]
plots(diags)

我们可以组合滤波器来得到更复杂的模式


In [ ]:
rots = straights + diags
corrs_cross = [correlate(cross, rot) for rot in rots]
plots(corrs_cross)

In [ ]:
rots = straights + diags
corrs = [correlate(images[5000], rot) for rot in rots]
plots(corrs)

接下来我们演示下采样的效果。
我们选择最基本的采样技术：__最大池化（max pooling）__。对大小为 ```7x7``` 的滑动窗口，只保留最大值。
__最大池化__ 是一个很实用的技巧，有几个好处：
- 因为它选取最大值，所以能保证对平移的不变性
- 缩小尺寸很有用，数据变得更紧凑、更容易比较
- 课程后面我们会看到，由于最大池化缩小了图像尺寸，网络后面执行的操作会有更大的感受野（对应输入图像中更大的区域），从而可以发现更高级的模式。


In [ ]:
import skimage

from skimage.measure import block_reduce

def pool(im): return block_reduce(im, (7,7), np.max)

plots([pool(im) for im in corrs])

现在我们用卷积来构建一个分类器。

为此，我们选取一组 _8_ 和 _1_ 的训练图片，用我们的滤波器组对它们做卷积，再做池化，并按类别和滤波器取平均。这样我们就得到了一组 _8_ 和 _1_ 的_代表性_签名。
给定一张新的测试图片，我们用同样的滤波器做卷积和池化计算它的特征，然后与_代表性_特征比较。选特征最_相似_的那个类别作为预测。


测试集保留 1000 张 _8_ 的图片，其余用于训练：我们把它们与滤波器组做卷积，对响应做最大池化，存到 ```pool8``` 里。


In [ ]:
pool8 = [np.array([pool(correlate(im, rot)) for im in eights[1000:]]) for rot in rots]

In [ ]:
len(pool8), pool8[0].shape

我们把前 5 张 _8_ 经过第一个滤波器和池化后的结果画出来。


In [ ]:
plots(pool8[0][0:5])

对集合里前 4 张 _8_，画出 8 个滤波器+池化的结果


In [ ]:
plots([pool8[i][0] for i in range(8)])
plots([pool8[i][1] for i in range(8)])
plots([pool8[i][2] for i in range(8)])
plots([pool8[i][3] for i in range(8)])

我们对数据做归一化，让激活值更平滑，并拉到一个相近的取值范围


In [ ]:
def normalize(arr): return (arr-arr.mean())/arr.std()

接下来，把每个滤波器来自 _rots_ 的所有响应取平均，得到平均 _8_。


In [ ]:
filts8 = np.array([ims.mean(axis=0) for ims in pool8])
filts8 = normalize(filts8)

我们应该能得到每个滤波器对应的、典型的 _8_ 响应。


In [ ]:
plots(filts8)

我们用 _1_ 类的训练样本做同样的事，并画出典型的 _1_。


In [ ]:
pool1 = [np.array([pool(correlate(im, rot)) for im in ones[1000:]]) for rot in rots]
filts1 = np.array([ims.mean(axis=0) for ims in pool1])
filts1 = normalize(filts1)

In [ ]:
plots(filts1)

你注意到 ```filts8``` 和 ```filts1``` 之间的区别了吗？是哪些？


我们定义一个函数，把给定图像与 ```rots``` 里所有滤波器做相关，并对响应做最大池化。


In [ ]:
def pool_corr(im): return np.array([pool(correlate(im, rot)) for rot in rots])

In [ ]:
plots(pool_corr(eights[1000]))

In [ ]:
#检查
plots([pool8[i][0] for i in range(8)])
np.allclose(pool_corr(eights[1000]),[pool8[i][0] for i in range(8)])

In [ ]:
# 用于投票式分类器的函数：根据 sse 距离指出
# 两个类别中最可能是哪一个
# n2 来自 norm2
# is8_n2 如果认为它是 8 就返回 1，否则返回 0
def is8_n2(im): return 1 if sse(pool_corr(im),filts1) > sse(pool_corr(im),filts8) else 0

我们做个检查，确认这个函数确实有效。把一张 _8_ 的图像分别与 ```filts8``` 和 ```filts1``` 做相关。对 _8_ 的距离应该更小。


In [ ]:
sse(pool_corr(eights[0]), filts8), sse(pool_corr(eights[0]), filts1)

In [ ]:
plot(eights[0])

现在我们在 1000 张 _8_ 和 1000 张 _1_ 的图片上测试我们的分类器


In [ ]:
nb_8_predicted_8, nb_1_predicted_8 = [np.array([is8_n2(im) for im in ims]).sum() for ims in [eights[:1000],ones[:1000]]]

nb_8_predicted_1, nb_1_predicted_1 = [np.array([(1-is8_n2(im)) for im in ims]).sum() for ims in [eights[:1000],ones[:1000]]]

In [ ]:
Precisionf_8, Recallf_8, Precisionf_1, Recallf_1 = compute_scores(nb_8_predicted_8,nb_8_predicted_1,nb_1_predicted_1,nb_1_predicted_8)

print('precision 8:', Precisionf_8, 'recall 8:', Recallf_8)
print('precision 1:', Precisionf_1, 'recall 1:', Recallf_1)
print('accuracy :', (Recallf_1+Recallf_8)/2)
print('accuracy baseline:', (Recall_1+Recall_8)/2)

在把嵌入尺寸从 $28\times 28 = 784$ 维向量降到 $4\times 4\times 8 = 128$ 维向量的同时，我们提高了准确率。

我们用一组预定义特征和一个训练样本集提取特征，成功构建了一个 _8_ 和 _1_ 的分类器。


## 5. 实操：用卷积神经网络改进分类

现在你将搭建一个能学习滤波器权重的神经网络。

网络的第一个层是一个卷积层，有 $8$ 个 $3\times 3$ 的滤波器。然后应用一个最大池化层，把图像尺寸降到 $4\times 4$，和我们上面做的一样。这会产生一个（展平后）大小为 $128 = 4\times 4\times 8$ 的向量。从这个向量出发，你需要预测对应的输入是 $1$ 还是 $8$。所以你又回到了前面课程里见过的分类问题。

你需要填写下面写的代码来构造你的 CNN。你需要在 Pytorch 文档里查找 [torch.nn](https://pytorch.org/docs/stable/nn.html) 的相关文档。


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class classifier(nn.Module):
    
    def __init__(self):
        super(classifier, self).__init__()
        # 填写下面缺失的条目
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=?)
        self.fc = nn.Linear(in_features=128, out_features=2)
        
    def forward(self,x):
        # 在这里实现你的网络，使用 F.max_pool2d、F.log_softmax，别忘了把向量展平
        x = self.conv1(x)
        #
        # 你的代码
        #
        #
        return x

In [ ]:
conv_class = classifier()

你的代码应该能在一个 3 张图片的 batch 上正常工作。


In [ ]:
batch_3images = train_set.data[0:2].type(torch.FloatTensor).resize_(3, 1, 28, 28)
#conv_class(batch_3images)

下面几行代码为训练集和测试集实现了数据加载器。不需要修改。


In [ ]:
bs = 64

l8 = np.array(0)
eights_dataset = [[torch.from_numpy(e.astype(np.float32)).unsqueeze(0), torch.from_numpy(l8.astype(np.int64))] for e in eights]
l1 = np.array(1)
ones_dataset = [[torch.from_numpy(e.astype(np.float32)).unsqueeze(0), torch.from_numpy(l1.astype(np.int64))] for e in ones]
train_dataset = eights_dataset[1000:] + ones_dataset[1000:]
test_dataset = eights_dataset[:1000] + ones_dataset[:1000]

train_loader = torch.utils.data.DataLoader(train_dataset,
    batch_size=bs, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,
    batch_size=bs, shuffle=True)

现在你需要编写训练循环。记录每个 epoch 的损失和准确率。


In [ ]:
def train(model,data_loader,loss_fn,optimizer,n_epochs=1):
    model.train(True)
    loss_train = np.zeros(n_epochs)
    acc_train = np.zeros(n_epochs)
    for epoch_num in range(n_epochs):
        running_corrects = 0.0
        running_loss = 0.0
        size = 0

        for data in data_loader:
            inputs, labels = data
            bs = labels.size(0)
            #
            #
            # 你的代码
            #
            #
            size += bs
        epoch_loss = running_loss.item() / size
        epoch_acc = running_corrects.item() / size
        loss_train[epoch_num] = epoch_loss
        acc_train[epoch_num] = epoch_acc
        print('Train - Loss: {:.4f} Acc: {:.4f}'.format(epoch_loss, epoch_acc))
    return loss_train, acc_train

In [ ]:
conv_class = classifier()
# 选择适当的损失
loss_fn = 
# 你的 SGD 优化器
learning_rate = 1e-3
optimizer_cl = 
# 然后训练 10 个 epoch
l_t, a_t = train(conv_class,train_loader,loss_fn,optimizer_cl,n_epochs = 10)

让我们再学 10 个 epoch


In [ ]:
l_t1, a_t1 = train(conv_class,train_loader,loss_fn,optimizer_cl,n_epochs = 10)

我们的网络看起来在学东西，但现在我们需要在测试集上检查它的准确率。


In [ ]:
def test(model,data_loader):
    model.train(False)

    running_corrects = 0.0
    running_loss = 0.0
    size = 0

    for data in data_loader:
        inputs, labels = data
            
        bs = labels.size(0)
        #
        # 你的代码
        #
        size += bs

    print('Test - Loss: {:.4f} Acc: {:.4f}'.format(running_loss / size, running_corrects.item() / size))

In [ ]:
test(conv_class,test_loader)

把优化器换成 Adam。


你的网络学了多少个参数？


你可以像下面这样查看它们：


In [ ]:
for m in conv_class.children():
    print('weights :', m.weight.data)
    print('bias :', m.bias.data)

In [ ]:
for m in conv_class.children():
    T_w = m.weight.data.numpy()
    T_b = m.bias.data.numpy()
    break

In [ ]:
plots([T_w[i][0] for i in range(8)])

In [ ]:
T_b

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)